# 00b · Dividir el CSV de DataCo en lotes
Lee `DataCoSupplyChainDataset.csv` del volumen `raw` y deja en `landing` cuatro carpetas con un `envios.csv` cada una:

| Lote | % | Contenido |
|---|---|---|
| `lote_01` | 70 % | carga inicial (pedidos más antiguos) |
| `lote_02` | 15 % | segunda carga |
| `lote_03` | 10 % | tercera carga |
| `lote_error` | 5 % | pedidos más recientes, sin la columna `Customer Id` (error provocado) |

`lote_01` es el CSV sin modificar. En `lote_02` y `lote_03` se cambia el `Customer Segment` de 20 clientes que ya estaban en `lote_01` (cambio simulado para que el SCD2 cierre versiones). Cada lote lleva una columna extra, `fecha_extraccion`.

In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.config import CATALOGO_POR_DEFECTO, VOLUMEN
from src.preparar_lotes import preparar

dbutils.widgets.text("catalogo", CATALOGO_POR_DEFECTO)
dbutils.widgets.text("csv_origen", "")   # vacío = /Volumes/<catalogo>/bronze/raw/DataCoSupplyChainDataset.csv
catalogo = dbutils.widgets.get("catalogo")
csv = dbutils.widgets.get("csv_origen") or f"/Volumes/{catalogo}/bronze/raw/DataCoSupplyChainDataset.csv"
salida = f"/Volumes/{catalogo}/bronze/{VOLUMEN}"

In [0]:
filas = preparar(csv, salida)
total = sum(filas.values())
for lote, n in filas.items():
    print(f"{lote:<11} {n:>8,} filas  ({100 * n / total:.1f} %)")

## Comprobación: carpetas creadas

In [0]:
for carpeta in sorted(dbutils.fs.ls(salida), key=lambda f: f.name):
    for f in dbutils.fs.ls(carpeta.path):
        print(f"{carpeta.name}/{f.name}  {f.size / 1024 / 1024:,.1f} MB")